# Random Forest Classification – Airline Customer Satisfaction Analysis

**Project**: Predict passenger satisfaction using survey data from Invistico Airlines.

In [ ]:
from google.colab import files
import pandas as pd

# Upload the CSV file
uploaded = files.upload()

# Load the dataset
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f"✅ Dataset loaded: {df.shape}")
print(df.head())
print("\nTarget Distribution:")
print(df['satisfaction'].value_counts())

In [ ]:
# Handle missing values
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())

# One-hot encode categorical features
df = pd.get_dummies(df, drop_first=True)

# Define features and target
X = df.drop('satisfaction_satisfied', axis=1)
y = df['satisfaction_satisfied']

print(f"Features after encoding: {X.shape[1]}")
print("✅ Preprocessing completed")

In [ ]:
from sklearn.model_selection import train_test_split

# 60% Train, 20% Validation, 20% Test
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)

print(f"Train set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Create PredefinedSplit
split_index = [-1] * len(X_train) + [0] * len(X_val)
ps = PredefinedSplit(split_index)

# Parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [15, 20, None],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

print("✅ Ready for GridSearchCV")

In [ ]:
# Grid Search
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1',
    cv=ps,
    n_jobs=-1,
    verbose=1
)

# Fit on train + val
grid_search.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

print("✅ Best Parameters:", grid_search.best_params_)
print("Best F1 Score (CV):", round(grid_search.best_score_, 4))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("=== Random Forest Test Performance ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("=== Decision Tree Test Performance ===")
print("Accuracy :", round(accuracy_score(y_test, y_pred_dt), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_dt), 4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("Top 10 Most Important Features:")
print(importances.head(10))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index)
plt.title('Top 10 Feature Importances - Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Executive Summary & Recommendations

**Random Forest significantly outperforms the single Decision Tree** by reducing overfitting and improving generalization.

### Key Drivers of Passenger Satisfaction:
- In-flight entertainment
- Online boarding experience
- Seat comfort
- WiFi service
- Cleanliness

**Recommendation to Management**:  
Focus resources on improving **digital services** (WiFi, online boarding) and **cabin comfort** to maximize customer satisfaction and loyalty.

### Explanation of F1-Score
The F1-score is the harmonic mean of Precision and Recall:  
**F1 = 2 × (Precision × Recall) / (Precision + Recall)**  
It is particularly useful when you need a balance between Precision and Recall, especially in imbalanced datasets.